# Canoas — Integração dos dados de campo (IFPC + Cubagem) com a nuvem LiDAR

Este notebook:

1. Carrega as três planilhas (`Cubagem`, `Dados_IFPC`, `Result_IFPC`) e o shapefile das parcelas.
2. Traduz os códigos de árvore (A, B, C... ) usando a legenda de campo.
3. Monta um mapa interativo (satélite/OSM) com as 24 parcelas marcadas, do jeito que o Google Maps mostraria.
4. Recorta a nuvem LiDAR em círculos de 400 m² (raio 11,28 m), no mesmo formato das parcelas de campo, gerando um arquivo `.las` por parcela — pronto para alimentar o FF3D / RayExtract.

> Edite os caminhos na célula **CONFIGURAÇÃO** antes de rodar.


In [24]:
# Se necessário, descomente para instalar as dependências
# !pip install geopandas shapely pyproj folium "laspy[lazrs]" nbformat --break-system-packages

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import laspy


## Configuração

In [25]:
# --- planilhas e shapefile (já enviados) ---
CUBAGEM_PATH      = "/mnt/scratch/matheuspimenta/euc/canoa/IFPC-Canoas/Cubagem_Canoas.xls"
DADOS_IFPC_PATH   = "/mnt/scratch/matheuspimenta/euc/canoa/IFPC-Canoas/Dados_IFPC_Canoas.xlsx"
RESULT_IFPC_PATH  = "/mnt/scratch/matheuspimenta/euc/canoa/IFPC-Canoas/Result_IFPC_Canoas.xlsx"
SHAPEFILE_PATH    = "/mnt/scratch/matheuspimenta/euc/canoa/IFPC-Canoas/shp-parcelas/"   # precisa dos .dbf/.shx/.prj junto na mesma pasta

# --- nuvem LiDAR (AJUSTE AQUI para o seu arquivo .las/.laz) ---
LIDAR_PATH = "/mnt/scratch/matheuspimenta/euc/canoa/LiDAR/T37_10cm/terra_las/cloud_merged.las"

# --- CRS dos dados de campo (confirmado no .prj do shapefile) ---
CRS_CAMPO = "EPSG:31983"  # SIRGAS 2000 / UTM 23S

# --- pasta de saída para as nuvens recortadas por parcela ---
OUT_DIR = "/mnt/scratch/matheuspimenta/euc/canoa/LiDAR/T37_10cm/parcelas"
os.makedirs(OUT_DIR, exist_ok=True)


## 1. Carregar as planilhas

Cada arquivo tem abas diferentes — carregamos tudo em dicionários de DataFrames para facilitar o acesso.

In [26]:
cubagem = pd.read_excel(CUBAGEM_PATH, sheet_name="Cubagem")

dados_ifpc = {
    "cadastro": pd.read_excel(DADOS_IFPC_PATH, sheet_name="Cadastro"),
    "dados":    pd.read_excel(DADOS_IFPC_PATH, sheet_name="Dados"),
}

result_ifpc = {
    "eq_hip":      pd.read_excel(RESULT_IFPC_PATH, sheet_name="Eq. Hip."),
    "eq_vol":      pd.read_excel(RESULT_IFPC_PATH, sheet_name="Eq. Vol."),
    "covas":       pd.read_excel(RESULT_IFPC_PATH, sheet_name="Covas"),
    "parcelas":    pd.read_excel(RESULT_IFPC_PATH, sheet_name="Parcelas"),
    "talhoes":     pd.read_excel(RESULT_IFPC_PATH, sheet_name="Talhões"),
    "estatistica": pd.read_excel(RESULT_IFPC_PATH, sheet_name="Estatística"),
}

print("Resumo dos dados carregados:")
for nome, df in {"cubagem": cubagem, **dados_ifpc, **result_ifpc}.items():
    print(f"  {nome:15s} -> {df.shape[0]:5d} linhas x {df.shape[1]:3d} colunas")


Resumo dos dados carregados:
  cubagem         ->   174 linhas x  14 colunas
  cadastro        ->     4 linhas x  10 colunas
  dados           ->  1316 linhas x  17 colunas
  eq_hip          ->     3 linhas x  21 colunas
  eq_vol          ->     3 linhas x  23 colunas
  covas           ->  1316 linhas x  27 colunas
  parcelas        ->    24 linhas x  85 colunas
  talhoes         ->     3 linhas x  89 colunas
  estatistica     ->     4 linhas x  39 colunas


## 2. Legenda dos códigos de árvore

Baseado na tabela de campo que você enviou. Aplicamos sobre a aba `Covas` (nível de fuste individual) e mostramos a distribuição.

In [27]:
LEGENDA_CODIGOS = {
    "A": "Árvore com DAP inferior a 4 cm - dominada",
    "B": "Árvore com bifurcação acima do DAP",
    "C": "Cancro",
    "D": "Árvore dominante",
    "E": "Remanescente de plantio anterior",
    "F": "Falha de plantio ou brotação",
    "G": "Galha",
    "H": "Árvore caída",
    "I": "Árvore inclinada",
    "J": "Broto ladrão",
    "L": "Bifurcação abaixo de 1,30 m (medir os dois diâmetros)",
    "M": "Árvore morta",
    "N": "Árvore normal",
    "O": "Quebra de ponteiro",
    "P": "\"Pau-preto\" ou gomose",
    "Q": "Árvore quebrada (se abaixo de 1,30 m, considerar como H)",
    "S": "Toco",
    "T": "Árvore com tortuosidade no fuste",
    "V": "Ponteiro seco",
    "W": "Touça morta",
    "X": "Ocorrência de incêndio/queimada",
}

covas = result_ifpc["covas"].copy()
covas["Descrição código"] = covas["Códigos"].map(LEGENDA_CODIGOS).fillna("(não catalogado na legenda)")

resumo_codigos = (
    covas["Códigos"]
    .value_counts()
    .rename_axis("Código")
    .reset_index(name="Contagem")
)
resumo_codigos["Descrição"] = resumo_codigos["Código"].map(LEGENDA_CODIGOS)
resumo_codigos = resumo_codigos.sort_values("Contagem", ascending=False).reset_index(drop=True)
resumo_codigos


,Código,Contagem,Descrição
0,L,470,"Bifurcação abaixo de 1,30 m (medir os dois diâ..."
1,N,440,Árvore normal
2,F,134,Falha de plantio ou brotação
3,D,96,Árvore dominante
4,S,59,Toco
5,I,54,Árvore inclinada
6,T,28,Árvore com tortuosidade no fuste
7,M,19,Árvore morta
8,B,7,Árvore com bifurcação acima do DAP
9,Q,3,"Árvore quebrada (se abaixo de 1,30 m, consider..."


In [28]:
# Códigos que indicam que NÃO existe fuste vivo/medível ali
# (ajuste essa lista se quiser um critério diferente)
CODIGOS_SEM_ARVORE = {"F", "M", "S", "W"}  # falha, morta, toco, touça morta

covas["arvore_valida"] = ~covas["Códigos"].isin(CODIGOS_SEM_ARVORE)

print(f"Fustes totais na base:                 {len(covas)}")
print(f"Fustes considerados 'árvore válida':    {covas['arvore_valida'].sum()}")
print(f"Fustes excluídos (falha/morta/toco):    {(~covas['arvore_valida']).sum()}")


Fustes totais na base:                 1316
Fustes considerados 'árvore válida':    1104
Fustes excluídos (falha/morta/toco):    212


In [29]:
# Contagem de árvores válidas por parcela — é o número que você vai comparar
# com a contagem de árvores detectadas pelo LiDAR dentro do mesmo círculo
contagem_por_parcela = (
    covas[covas["arvore_valida"]]
    .groupby("PAR_CODIGO")
    .size()
    .rename("n_arvores_campo")
    .reset_index()
    .rename(columns={"PAR_CODIGO": "Parcela"})
)
contagem_por_parcela


,Parcela,n_arvores_campo
0,500,41
1,503,36
2,508,39
3,509,24
4,594,47
5,602,45
6,611,55
7,612,48
8,621,53
9,622,45


## 3. Shapefile das parcelas + estatísticas

In [30]:
parcelas_gdf = gpd.read_file(SHAPEFILE_PATH)
parcelas_gdf["Parcela"] = parcelas_gdf["Parcela"].astype(int)

if parcelas_gdf.crs is None:
    parcelas_gdf = parcelas_gdf.set_crs(CRS_CAMPO)
print("CRS do shapefile:", parcelas_gdf.crs)

raio_por_parcela = dados_ifpc["dados"][["Parcela", "Raio Parcela"]].drop_duplicates()

parcelas_full = (
    parcelas_gdf
    .merge(result_ifpc["parcelas"], on="Parcela", how="left", suffixes=("", "_res"))
    .merge(raio_por_parcela, on="Parcela", how="left")
    .merge(contagem_por_parcela, on="Parcela", how="left")
)

# coordenadas UTM (para o recorte do LiDAR) e lat/lon (para o mapa)
parcelas_full["x_utm"] = parcelas_full.geometry.x
parcelas_full["y_utm"] = parcelas_full.geometry.y

parcelas_wgs84 = parcelas_full.to_crs(epsg=4326)
parcelas_full["lat"] = parcelas_wgs84.geometry.y
parcelas_full["lon"] = parcelas_wgs84.geometry.x

parcelas_full[["Talhao", "Parcela", "x_utm", "y_utm", "lat", "lon",
               "Raio Parcela", "n_arvores_campo",
               "Dap.médio (cm)", "HT.média (m)", "VTCC (m³/ha)"]]


CRS do shapefile: EPSG:31983


,Talhao,Parcela,x_utm,y_utm,lat,lon,Raio Parcela,n_arvores_campo,Dap.médio (cm),HT.média (m),VTCC (m³/ha)
0,062,500,443180.0,8007278.0,-18.022182,-45.536794,11.28,41,15.2,25.9,218.32
1,062,503,443187.0,8007362.0,-18.021423,-45.536726,11.28,36,16.4,26.2,223.43
2,062,508,443160.0,8007506.0,-18.020121,-45.536977,11.28,39,15.0,25.4,200.72
3,062,509,443319.0,8007517.0,-18.020025,-45.535474,11.28,24,17.6,27.9,162.69
4,037,594,441497.0,8012311.0,-17.976648,-45.552552,11.28,47,13.8,23.0,193.95
5,037,602,441641.0,8012400.0,-17.975847,-45.551189,11.28,45,14.2,22.7,192.89
6,037,611,441698.0,8012600.0,-17.974041,-45.550645,11.28,55,13.3,23.1,202.06
7,037,612,441858.0,8012571.0,-17.974307,-45.549135,11.28,48,14.2,22.3,195.82
8,037,621,441708.0,8012708.0,-17.973065,-45.550548,11.28,53,12.9,22.0,203.73
9,037,622,441853.0,8012691.0,-17.973223,-45.549179,11.28,45,14.4,22.3,190.92


## 4. Mapa com as parcelas marcadas (estilo Google Maps)

Uso o Esri World Imagery como camada de satélite (não precisa de chave de API, ao contrário do Google Maps).
Cada círculo tem o raio real da parcela (11,28 m) na escala do mapa.

In [31]:
center_lat = parcelas_full["lat"].mean()
center_lon = parcelas_full["lon"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles=None)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Satélite (Esri)",
    overlay=False,
).add_to(m)
folium.TileLayer("OpenStreetMap", name="Mapa (OSM)", overlay=False).add_to(m)

paleta = ["#e6194B", "#3cb44b", "#4363d8", "#f58231", "#911eb4"]
cores_talhao = {t: paleta[i % len(paleta)] for i, t in enumerate(sorted(parcelas_full["Talhao"].unique()))}

for _, row in parcelas_full.iterrows():
    popup_html = (
        f"<b>Talhão:</b> {row['Talhao']}<br>"
        f"<b>Parcela:</b> {row['Parcela']}<br>"
        f"<b>Árvores válidas (campo):</b> {row['n_arvores_campo']}<br>"
        f"<b>DAP médio:</b> {row['Dap.médio (cm)']:.1f} cm<br>"
        f"<b>Altura média:</b> {row['HT.média (m)']:.1f} m<br>"
        f"<b>VTCC:</b> {row['VTCC (m³/ha)']:.1f} m³/ha"
    )
    folium.Circle(
        location=[row["lat"], row["lon"]],
        radius=row["Raio Parcela"],
        color=cores_talhao[row["Talhao"]],
        fill=True,
        fill_opacity=0.25,
        weight=2,
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"Parcela {row['Parcela']} (Talhão {row['Talhao']})",
    ).add_to(m)

folium.LayerControl().add_to(m)
m.save("mapa_parcelas_canoas.html")
m  # exibe o mapa inline no notebook


## 5. Recortar a nuvem LiDAR no mesmo círculo de cada parcela

Gera um `.las` por parcela, contendo só os pontos dentro do buffer de 11,28 m em volta do centro GPS da parcela — exatamente a mesma área que os medidores de campo cobriram.

⚠️ **Importante**: a nuvem LiDAR precisa estar no mesmo CRS dos dados de campo (`EPSG:31983`). Se estiver em outro sistema, reprojete antes (com `pdal` ou `laspy` + `pyproj`) ou ajuste `CRS_CAMPO`/`CRS_LIDAR` abaixo.

In [32]:
def recortar_parcela_las(las_path, centro_x, centro_y, raio, out_path):
    """Recorta um LAS/LAZ mantendo só os pontos dentro do círculo
    (centro_x, centro_y, raio), nas mesmas unidades do CRS do arquivo.
    Salva em out_path e retorna o número de pontos recortados."""
    with laspy.open(las_path) as f:
        las = f.read()

    dist2 = (las.x - centro_x) ** 2 + (las.y - centro_y) ** 2
    mask = dist2 <= raio ** 2

    n_pontos = int(mask.sum())
    if n_pontos == 0:
        return 0

    las_clip = laspy.LasData(las.header)
    las_clip.points = las.points[mask].copy()
    las_clip.write(out_path)
    return n_pontos


In [33]:
resultados_recorte = []

for _, row in parcelas_full.iterrows():
    talhao = row["Talhao"]
    parcela = row["Parcela"]
    out_path = os.path.join(OUT_DIR, f"talhao_{talhao}_parcela_{parcela}.las")

    n_pontos = recortar_parcela_las(
        LIDAR_PATH,
        centro_x=row["x_utm"],
        centro_y=row["y_utm"],
        raio=row["Raio Parcela"],
        out_path=out_path,
    )
    resultados_recorte.append({
        "Talhao": talhao,
        "Parcela": parcela,
        "n_pontos_lidar": n_pontos,
        "n_arvores_campo": row["n_arvores_campo"],
        "arquivo": out_path if n_pontos > 0 else None,
    })

recortes_df = pd.DataFrame(resultados_recorte)
recortes_df


,Talhao,Parcela,n_pontos_lidar,n_arvores_campo,arquivo
0,062,500,0,41,None
1,062,503,0,36,None
2,062,508,0,39,None
3,062,509,0,24,None
4,037,594,129918,47,/mnt/scratch/matheuspimenta/euc/canoa/LiDAR/T3...
5,037,602,123065,45,/mnt/scratch/matheuspimenta/euc/canoa/LiDAR/T3...
6,037,611,277233,55,/mnt/scratch/matheuspimenta/euc/canoa/LiDAR/T3...
7,037,612,175439,48,/mnt/scratch/matheuspimenta/euc/canoa/LiDAR/T3...
8,037,621,168327,53,/mnt/scratch/matheuspimenta/euc/canoa/LiDAR/T3...
9,037,622,156619,45,/mnt/scratch/matheuspimenta/euc/canoa/LiDAR/T3...


### Checagem rápida — densidade de pontos por parcela

Não confirma contagem de árvores (isso só a rede neural faz), mas ajuda a identificar parcelas com nuvem
rala/vazia antes de rodar o FF3D/RayExtract nelas.

In [34]:
recortes_df["pontos_por_m2"] = recortes_df["n_pontos_lidar"] / (np.pi * 11.28**2)
recortes_df.sort_values("pontos_por_m2")


,Talhao,Parcela,n_pontos_lidar,n_arvores_campo,arquivo,pontos_por_m2
0,062,500,0,41,None,0.000000
1,062,503,0,36,None,0.000000
2,062,508,0,39,None,0.000000
3,062,509,0,24,None,0.000000
15,034,649,0,41,None,0.000000
18,034,660,0,57,None,0.000000
17,034,659,0,54,None,0.000000
16,034,650,0,43,None,0.000000
23,034,697,0,50,None,0.000000
22,034,684,0,59,None,0.000000


## Próximos passos

- Rode o FF3D e o RayExtract em cada arquivo de `parcelas_lidar_recortadas/`.
- Some as contagens de árvores detectadas por parcela e compare com `n_arvores_campo` (coluna já calculada acima).
- Para o pareamento individual (DAP/altura), veja a sugestão de matching por menor custo (tipo algoritmo húngaro) discutida na conversa — pode ser adicionado numa próxima célula quando as saídas do FF3D/RayExtract estiverem prontas.
